# 104장 · 답변 카드와 요청 대기 상태

답변 카드는 현재 질문의 답과 원문 위치를 함께 표시합니다. 새 요청을 보낸 직후에는 이전 카드 중 무엇을 비워야 하는지 정합니다.

102장의 응답으로 장비 A의 답과 근거 위치를 한 화면에 배치합니다. 결과 카드와 응답 대기 상태에서 사용자에게 보이는 정보의 차이를 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 본문과 원문 위치의 표시 규칙

완성 화면 시안　상단에 “장비 A의 점검 전 확인 사항은?”을 두고 그 아래에 “점검 전 전원 표시등을 확인합니다.”라고 씁니다. 바로 아래 ‘근거 문서’ 영역에는 자료명과 페이지·절·개정일·버전을 둡니다. 원문 바로가기 기능을 구현하는 팀은 주소를 문서 목록에서 얻고 해당 절이 열리는지 직접 확인해야 합니다.


### 셀 01 · 결과 카드의 근거 위치

로컬 예제의 Q01 응답에서 현재 답과 첫 출처를 읽어 화면 카드에 들어갈 문장을 만듭니다.

카드에는 답변과 함께 문서 이름·쪽·절·개정일을 표시합니다. RULE-A는 근거를 다시 찾을 때 쓰는 문서 ID이므로 출처 줄에도 보이게 합니다. model_calls 같은 실행 정보는 개발 기록에 남깁니다.


In [1]:
result = {'status': 'answered',
 'answer': '점검 전 전원 표시등을 확인합니다.',
 'sources': [{'id': 'RULE-A',
              'text': '점검 전 전원 표시등을 확인합니다.',
              'source': '가상 장비실 규정 A',
              'page': 1,
              'section': '2.1',
              'revised_on': '2026-11-01',
              'version': 'rules-v1'}],
 'mode': 'local_rule_fixture',
 'model_calls': 0}
source = result["sources"][0]
print(result["answer"])
print(f'근거 문서 · {source["id"]} · {source["source"]} · {source["page"]}쪽 {source["section"]}절')
print(f'개정일 {source["revised_on"]} · {source["version"]}')

점검 전 전원 표시등을 확인합니다.
근거 문서 · RULE-A · 가상 장비실 규정 A · 1쪽 2.1절
개정일 2026-11-01 · rules-v1


규정 A, 1쪽 2.1절과 개정일이 답 바로 아래에 보입니다. 줄을 나눠 놓으면 작은 화면에서도 문서 위치와 답의 문장을 따로 읽을 수 있습니다.

사용자는 답을 읽은 뒤 이 지시가 어느 판본에 있는지 확인해야 합니다. 카드에서 규정 이름·쪽·절·개정일을 한 답과 묶어 표시하면 다른 검색 결과의 근거를 잘못 따라갈 가능성을 줄일 수 있습니다. 응답 대기 중에는 이전 카드의 근거를 붙이지 않습니다.


### 답과 인용 위치가 같은 응답에서 왔는지 확인하기

출력의 첫 줄은 result의 answer, 다음 두 줄은 같은 result의 첫 번째 source에서 나옵니다. 새 질문을 보낸 뒤 답만 바꾸고 이전 source를 재사용하면 새 답 옆에 옛 규정이 붙습니다. 한 응답에서 답과 출처를 함께 꺼내는 이유입니다.


## 대기 중 이전 답변과 출처 초기화

시안의 상태 변화　보내기 전에는 질문 입력과 전송을 허용합니다. 보낸 직후에는 “응답을 기다리는 중”과 현재 요청 표식을 보여 주고 이전 답은 과거 대화에 둡니다. 새 응답을 받고 나서야 현재 답과 출처를 넣습니다. 지난 문장을 그대로 둔다면 사용자가 기다리는 중에 그 답을 새 결과로 읽을 수 있습니다.


#### 요청 대기와 완료 화면의 차이

<div class="demo-panels"><div><small>① 전송 직후 · next-01</small><h4>응답을 받은 뒤의 결과 카드</h4><p>질문을 보냈습니다. 응답을 기다리는 중입니다.</p><p>출처 0건</p></div><div><small>② 현재 요청의 응답 도착</small><h4>응답을 받은 뒤의 결과 카드</h4><p>점검 전 전원 표시등을 확인합니다.</p><p><b>근거</b>　RULE-A · 가상 장비실 규정 A · 1쪽 2.1절<br>개정일 2026-11-01 · rules-v1</p></div></div><p>화면 시안입니다. 왼쪽에는 이전 답의 출처가 없고 오른쪽에는 지금 읽은 절이 답 바로 아래에 나타납니다.</p>


### 과제 02 · 선택 연습 · 대기 상태 표시

현재 답을 아직 받지 못한 경우를 담아 대기 문구를 만들어 보세요. 앞 질문의 출처를 새 카드에 복사하지 않습니다.


In [2]:
# 현재 요청 ID, 대기 문구, 비어 있는 새 출처 목록을 작성하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>view = {&quot;request_id&quot;: &quot;next-01&quot;, &quot;message&quot;: &quot;질문을 보냈습니다. 응답을 기다리는 중입니다.&quot;, &quot;sources&quot;: []}
print(view[&quot;message&quot;], view[&quot;sources&quot;])</code></pre>
<p>실행 결과</p><pre><code>질문을 보냈습니다. 응답을 기다리는 중입니다. []</code></pre>
<p>지금은 진행 안내와 빈 출처만 보입니다. 팀 화면에서는 이 안내를 어디에 둘지, 단축키나 읽기 도구로도 상태를 알 수 있을지 정할 수 있습니다.</p>
</details>


#### 새 요청의 대기 화면

<mark>아직 응답이 없을 때 직전 답변의 출처를 현재 질문에 붙이지 않습니다.</mark> 정상 답변 뒤 다른 요청을 보내고 대기 중인 순간의 제목·문구·출처를 읽으세요. 이전 대화 기록은 보존하면서 현재 카드만 초기화되는지 확인합니다.

선수 개념 · 97장 대화 턴의 현재 출처와 이전 기록

## 늦게 도착한 응답이 새 화면을 덮어쓰지 못하게 하기

검색 버튼을 두 번 눌렀고 첫 번째 응답이 더 늦게 도착했습니다. 기다리는 동안 새 출처 목록을 비웠더라도, 이전 응답이 마지막에 그 목록을 다시 채울 수 있습니다.

### 셀 04 · 응답의 요청 ID와 현재 화면의 요청 ID 대조

r-A 다음에 r-B를 보냈다고 가정합니다. 네트워크 대신 호출 순서를 직접 정해 두 번째 화면에서 첫 답변이 무시되는지 확인합니다.

In [3]:
def apply_response(view, request_id, answer, sources):
    if view["active_request"] != request_id:
        return view
    return {**view, "pending": False, "answer": answer,
            "sources": [item["id"] for item in sources]}

view = {"active_request": "r-B", "pending": True, "answer": "", "sources": []}
ignored = apply_response(view, "r-A", result["answer"], result["sources"])
print(ignored["pending"], ignored["sources"])
finished = apply_response(ignored, "r-B", "점검 전 고정 장치를 확인합니다.",
                          [{"id": "RULE-B"}])
print(finished["pending"], finished["sources"])

True []
False ['RULE-B']


첫 응답을 받은 뒤에도 r-B는 기다리는 중이고 출처가 비어 있습니다. 두 번째 응답이 도착한 뒤에야 RULE-B를 그립니다. 과거 r-A의 내용을 대화 이력에 저장할지는 별도의 저장 정책입니다.

### 팀 화면에서 같은 순서를 재현하기

두 입력의 응답 도착 순서를 바꿔 보세요. 두 번째 요청이 먼저 끝난 뒤 첫 번째 요청이 도착하는 경우에도 완료된 카드와 출처가 B로 남아야 합니다. 화면의 로딩 표시와 출처를 같은 요청 ID에 묶어 검사합니다.

<mark>마지막에 도착한 메시지가 지금 사용자가 기다리는 요청의 메시지인지 확인합니다.</mark> 응답 시각이 늦어도 그 답변이 가리키는 요청은 r-A입니다. r-B 화면에는 r-B의 자료만 표시합니다.

105장은 현재 요청의 결과가 검색 근거 없음, 저장 실패, 전달 여부 미확인일 때 안내 문구를 정합니다.

## 돌아보기

105장에는 완성된 답변 화면과, 보내는 중에는 어떤 항목이 비어 있어야 하는지 넘기세요. 노트북 출력은 화면 문안의 미리보기이며 브라우저에서 버튼을 누른 관찰 기록은 아닙니다.
